In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "tauzin2020context")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "tauzin_et_al_2020_from_manuel_bohn_not_original.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="tauzin2020context"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"individual": "ape",
    "species":"species_original"}, inplace=True)

df['ape'] = df['ape'].str.rstrip()



In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')


In [4]:
df['experiment'] = df['experiment'].str.strip('experiment')

In [5]:
condition=[]
for index, row in df.iterrows():
    if not pd.isna(row['condition_1']):
        condition.append(row['condition_1'])
    else:
        condition.append(row['condition_2'])
df = df.assign(condition=condition)
# df.columns
df.rename(columns={"ape": "participant", "food_position":"food_position_center"}, inplace=True)

In [6]:
fulldf=df[['study_id', 'experiment', 'participant','sex','species',  'session', 'trial',
        'condition', 'experimenter', 'position_hole', 'hq_food', 
        'indicated', 'bent', 'plexi', 'modified_point', 'food_position_center', 'lateral_point'  ]]


In [7]:
# fulldf['ape'].unique()

In [8]:
# fulldf['experiment'].unique()

In [9]:
for index in range(1,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'tauzin2020context_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'tauzin2020context_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

